# P2 — Analyse exploratoire : Mobilité urbaine (CityBikes API v2 + Historique 5 ans)

**Responsable : P2**

Ce notebook exploite directement les fichiers Parquet de la couche Silver (`silver/velos/` et `silver/velos_historique/` sur MinIO ou en local) pour répondre aux **5 questions métier** :
1. Quelles stations présentent le plus fort taux d’utilisation ?
2. Peut-on identifier des zones où l’offre de vélos est insuffisante ?
3. Quels sont les pics d’utilisation journaliers du réseau de vélos ?
4. Existe-t-il des déséquilibres géographiques dans la disponibilité des vélos ?
5. Peut-on détecter des stations critiques nécessitant un rééquilibrage des vélos ?

In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px

from common.config import get_settings, load_ref_ville
from common.storage import get_storage
from common.warehouse import try_get_warehouse
from p2_velos.analytics import (
    daily_usage_peaks,
    detect_critical_stations,
    detect_undersupplied_zones,
    ensure_enriched_dataframe,
    geographic_imbalance,
    top_utilisation_stations,
)

settings = get_settings()
ref_villes = load_ref_ville()
city_centers = {v['ville_id']: (v['lat'], v['lon']) for v in ref_villes}
USE_CLICKHOUSE = False
wh = try_get_warehouse()
print(
    f"Villes configurées ({len(ref_villes)}) | MinIO S3 : {settings.minio_endpoint} "
    f"(buckets: {settings.minio_bucket_bronze}, {settings.minio_bucket_silver}, {settings.minio_bucket_gold}) | "
    f"ClickHouse : {'connecté' if wh is not None else 'hors ligne'}"
)

Villes configurées (12) | MinIO S3 : localhost:9000 (buckets: bronze, silver, gold) | ClickHouse : connecté


## 0. Contrôle de la couverture temporelle et volumétrie (fenêtre 5 ans)

> Si seule la journée en cours apparaît, chargez l'historique officiel `data.citybik.es` en Silver Parquet via :
> `python -m p2_velos.backfill --years 5` (ou `--max-months-per-city 1` pour un premier mois rapide sur les 12 villes).

In [12]:
df_local = pd.DataFrame()
source_label = 'Aucune donnée'

# 1. Connexion prioritaire à MinIO S3 (bucket silver : s3://silver/velos/ & s3://silver/velos_historique/)
frames = []
try:
    st_minio = get_storage(use_local=False)
    velo_keys = [
        k
        for prefix in ('silver/velos/', 'silver/velos_historique/')
        for k in st_minio.list(prefix)
        if 'velo_releve_' in k and k.endswith('.parquet')
    ]
    for k in velo_keys:
        frames.append(st_minio.get_parquet(k))
    if frames:
        df_local = ensure_enriched_dataframe(
            pd.concat(frames, ignore_index=True).drop_duplicates(
                subset=['ville_id', 'station_id', 'timestamp'], keep='last'
            )
        )
        source_label = f'MinIO S3 (s3://silver/velos* — {len(velo_keys)} fichiers Parquet)'
except Exception as exc:
    print(f'MinIO indisponible ({exc}), tentative ClickHouse / local...')

# 2. Repli ClickHouse Lakehouse si aucun Parquet dans MinIO
if df_local.empty and wh is not None:
    try:
        df_ch = wh.query_df('SELECT * FROM urbanhub.velo_releve FINAL')
        if not df_ch.empty:
            df_local = ensure_enriched_dataframe(df_ch)
            source_label = 'ClickHouse Lakehouse (urbanhub.velo_releve)'
    except Exception:
        pass

# 3. Repli disque local (data/silver/velos*)
if df_local.empty:
    base_silver = Path(settings.local_data_dir) / 'silver'
    local_frames = []
    for sub in ('velos', 'velos_historique'):
        d = base_silver / sub
        if d.exists():
            for p in sorted(d.rglob('velo_releve_*.parquet')):
                local_frames.append(pd.read_parquet(p))
    if local_frames:
        df_local = ensure_enriched_dataframe(
            pd.concat(local_frames, ignore_index=True).drop_duplicates(
                subset=['ville_id', 'station_id', 'timestamp'], keep='last'
            )
        )
        source_label = 'Parquet Silver local (data/silver/velos*)'

if not df_local.empty:
    print(
        f'Source : {source_label} | {len(df_local):,} relevés | '
        f'{df_local["ville_id"].nunique()} villes | '
        f'Période UTC : {df_local["timestamp"].min()} -> {df_local["timestamp"].max()}'
    )
    summary_city = (
        df_local.groupby('ville_id', as_index=False)
        .agg(
            nb_stations=('station_id', 'nunique'),
            nb_releves=('timestamp', 'count'),
            debut_utc=('timestamp', 'min'),
            fin_utc=('timestamp', 'max'),
            velos_moyens=('bikes_available', lambda s: round(float(s.mean()), 1)),
            taux_utilisation_pct=('taux_utilisation', lambda s: round(float(s.mean() * 100), 1)),
            pct_critiques=('est_critique', lambda s: round(float(s.mean() * 100), 1)),
        )
        .sort_values('nb_releves', ascending=False)
    )
    display(summary_city)
else:
    print('Aucune donnée Silver vélos trouvée dans MinIO, ClickHouse ou data/silver/.')

Source : MinIO S3 (s3://silver/velos* — 4 fichiers Parquet) | 11,663 relevés | 12 villes | Période UTC : 2026-11-20 15:03:01.824212+00:00 -> 2026-11-20 15:14:30.242986+00:00


,ville_id,nb_stations,nb_releves,debut_utc,fin_utc,velos_moyens,taux_utilisation_pct,pct_critiques
7,paris,1513,4539,2026-11-20 15:03:01.824212+00:00,2026-11-20 15:11:24.692465+00:00,12.5,60.7,36.0
11,toulouse,454,1816,2026-11-20 15:05:59.194149+00:00,2026-11-20 15:13:51.125977+00:00,5.9,65.0,25.8
3,lyon,456,1368,2026-11-20 15:03:56.754833+00:00,2026-11-20 15:13:11.276081+00:00,5.1,74.5,42.6
2,lille,265,1060,2026-11-20 15:05:31.174703+00:00,2026-11-20 15:14:30.242986+00:00,6.4,67.4,20.5
0,bordeaux,219,876,2026-11-20 15:05:28.942525+00:00,2026-11-20 15:14:28.718792+00:00,6.8,65.6,13.5
9,rouen,127,508,2026-11-20 15:05:26.921059+00:00,2026-11-20 15:14:26.920420+00:00,5.3,58.2,15.2
4,marseille,123,492,2026-11-20 15:05:29.491586+00:00,2026-11-20 15:14:28.662930+00:00,0.0,100.0,100.0
1,clermont-ferrand,82,328,2026-11-20 15:05:26.593705+00:00,2026-11-20 15:14:25.944034+00:00,5.8,71.8,19.8
8,rennes,57,228,2026-11-20 15:05:17.674353+00:00,2026-11-20 15:14:17.994482+00:00,8.1,66.0,12.3
5,montpellier,48,192,2026-11-20 15:05:26.641515+00:00,2026-11-20 15:14:25.947366+00:00,1.9,87.2,33.3


### Échantillon des 7 variables obligatoires dans la couche Silver (`velo_releve`)

In [13]:
required_cols = ['station_id', 'station_name', 'latitude', 'longitude', 'bikes_available', 'free_slots', 'timestamp']
if USE_CLICKHOUSE:
    sample_df = wh.query_df(
        'SELECT ville_id, station_id, station_name, latitude, longitude, bikes_available, free_slots, timestamp '
        'FROM urbanhub.velo_releve ORDER BY timestamp DESC LIMIT 10'
    )
else:
    sample_df = df_local[['ville_id'] + required_cols].head(10)
expected_cols = ['ville_id'] + required_cols

# ClickHouse may return an empty DataFrame without schema when no rows match.
if sample_df.empty and len(sample_df.columns) == 0:
    sample_df = pd.DataFrame(columns=expected_cols)

assert all(c in sample_df.columns for c in required_cols)
sample_df

,ville_id,station_id,station_name,latitude,longitude,bikes_available,free_slots,timestamp
0,paris,0003131bffcf527ba3a560fc46d38b6f,Argenson - Château,48.888559,2.264200,5,19,2026-11-20 15:03:01.824887+00:00
1,paris,000471e1f2b458a6e68a8ae732880557,Saint Lambert - Blomet,48.836591,2.293056,2,24,2026-11-20 15:03:01.921449+00:00
2,paris,001e872d5c8bb1d9d69593da89a6da9e,Saint-Maur - République,48.864186,2.378317,1,48,2026-11-20 15:03:01.831954+00:00
3,paris,0028fad7cbbbe461aa18ae720500f8ef,Mairie de Rosny-sous-Bois,48.871257,2.486581,22,7,2026-11-20 15:03:01.826374+00:00
4,paris,003fde84c86105536fa7bc92ce5bd8a9,Labouret - Saint-Denis,48.924067,2.253916,20,3,2026-11-20 15:03:01.829446+00:00
5,paris,00718dcc2fa34b019d9cba625c94b1e7,Mouton Duvernet - Général Leclerc,48.831633,2.329312,1,27,2026-11-20 15:03:01.829565+00:00
6,paris,009ceb0db951391aa947e5c2d939518e,Pernety - Raymond Losserand,48.834255,2.317609,3,29,2026-11-20 15:03:01.927031+00:00
7,paris,00cd144dd6914f51a44cbfad2c12984f,Champs de Manoeuvre - Cartoucherie,48.835261,2.447974,32,2,2026-11-20 15:03:01.927905+00:00
8,paris,011948af9fad35dfda6e1adf45b81bed,Belleville - Télégraphe,48.875605,2.399555,3,38,2026-11-20 15:03:01.923153+00:00
9,paris,012b17de09d1300f63c8b63067b39063,Abbeville - Faubourg Poissonnière,48.879223,2.349147,0,14,2026-11-20 15:03:01.927783+00:00


## Question 1 : Quelles stations présentent le plus fort taux d’utilisation ?

In [14]:
if USE_CLICKHOUSE:
    top_stations = wh.query_df('''
        SELECT
            ville_id,
            station_name,
            capacite_moyenne,
            round(taux_utilisation_moyen * 100, 1) AS taux_utilisation_pct,
            volatilite_velos,
            amplitude_velos,
            nb_releves
        FROM urbanhub.v_velo_utilisation_stations
        ORDER BY taux_utilisation_moyen DESC, volatilite_velos DESC, capacite_moyenne DESC
        LIMIT 20
    ''')
else:
    top_stations = top_utilisation_stations(df_local, top_n=20)

fig1 = px.bar(
    top_stations.sort_values('taux_utilisation_pct'),
    x='taux_utilisation_pct',
    y='station_name',
    color='ville_id',
    orientation='h',
    hover_data=['capacite_moyenne', 'volatilite_velos', 'amplitude_velos', 'nb_releves'],
    title="Top 20 des stations présentant le plus fort taux d'utilisation",
    labels={'taux_utilisation_pct': "Taux d'utilisation moyen (%)", 'station_name': 'Station'},
    height=600,
)
fig1.show()
top_stations

,ville_id,station_id,station_name,latitude,longitude,capacite_moyenne,velos_dispo_moyens,taux_utilisation_moyen,taux_occupation_moyen,volatilite_velos,min_velos,max_velos,rotation_totale,nb_releves,amplitude_velos,taux_utilisation_pct,taux_occupation_pct
2472,paris,d3bd04f60bec2a43506568b72170354f,Invalides - Quai d'Orsay,48.862523,2.310176,400.0,0.0,1.0,0.0,0.0,0,0,0.0,3,0,100.0,0.0
2674,paris,f72384ff88bee74a7fb3da49e1367f16,Concorde - Place de la Madeleine,48.870621,2.324220,400.0,0.0,1.0,0.0,0.0,0,0,0.0,3,0,100.0,0.0
1645,paris,4837dabfb36f1567edcef37a72bf4a22,Station Tour de France,48.883291,2.334151,200.0,0.0,1.0,0.0,0.0,0,0,0.0,3,0,100.0,0.0
1119,marseille,d74960ffcea6c80ec1ed6d681c9d259c,Escale Borély,43.254948,5.376452,60.0,0.0,1.0,0.0,0.0,0,0,0.0,4,0,100.0,0.0
2486,paris,d5f5b73d5f0c3069a29370f1b343f0ac,Herschel - Observatoire,48.843424,2.337665,56.0,0.0,1.0,0.0,0.0,0,0,0.0,3,0,100.0,0.0
1551,paris,37a2ba913c8152ebeddb3228fb210d15,Filles Saint-Thomas - Place de la Bourse,48.868900,2.339754,54.0,0.0,1.0,0.0,0.0,0,0,0.0,3,0,100.0,0.0
1279,paris,08cd405dc9467f0d86e2b6717e1382d3,Raspail - Bac,48.855629,2.325660,52.0,0.0,1.0,0.0,0.0,0,0,0.0,3,0,100.0,0.0
2031,paris,8d249a46e5837f8bfcc93e5554e19b97,Porte d'Orléans,48.822764,2.324931,47.0,0.0,1.0,0.0,0.0,0,0,0.0,3,0,100.0,0.0
2622,paris,eeae167c8e80e84399d098767f7d1f50,Rougemont - Poissonnière,48.871612,2.345873,44.0,0.0,1.0,0.0,0.0,0,0,0.0,3,0,100.0,0.0
1033,marseille,1305368b25e2af76a084edccdffd8abe,Rond Point du Prado,43.273128,5.390727,40.0,0.0,1.0,0.0,0.0,0,0,0.0,4,0,100.0,0.0


## Question 2 : Peut-on identifier des zones où l’offre de vélos est insuffisante ?

In [15]:
if USE_CLICKHOUSE:
    zones_sous_offre = wh.query_df('''
        SELECT
            ville_id,
            round(latitude, 2) AS zone_lat,
            round(longitude, 2) AS zone_lon,
            uniqExact(station_id) AS nb_stations,
            any(station_name) AS exemple_station,
            round(avg(bikes_available), 2) AS velos_dispo_moyens,
            round(countIf(est_vide = 1) / count() * 100, 1) AS pct_temps_vide,
            round(countIf(taux_occupation <= 0.15) / count() * 100, 1) AS pct_temps_sous_offre,
            count() AS nb_releves
        FROM urbanhub.velo_releve
        GROUP BY ville_id, zone_lat, zone_lon
        ORDER BY pct_temps_sous_offre DESC, pct_temps_vide DESC, nb_stations DESC
        LIMIT 100
    ''')
else:
    zones_sous_offre = detect_undersupplied_zones(df_local, grid_precision=2, low_supply_threshold=0.15)

fig2 = px.scatter_map(
    zones_sous_offre.head(100),
    lat='zone_lat',
    lon='zone_lon',
    size='nb_stations',
    color='pct_temps_sous_offre',
    color_continuous_scale='YlOrRd',
    hover_name='exemple_station',
    hover_data=['ville_id', 'nb_stations', 'velos_dispo_moyens', 'pct_temps_vide'],
    map_style='carto-positron',
    zoom=5,
    height=520,
    title='Zones à offre de vélos insuffisante (<= 15 % de vélos disponibles)',
)
fig2.show()
zones_sous_offre.head(15)

,ville_id,zone_lat,zone_lon,nb_stations,exemple_station,capacite_totale,velos_dispo_moyens,taux_occupation_moyen,pct_temps_vide,pct_temps_sous_offre,nb_releves,taux_occupation_pct
136,lille,50.61,3.05,1,Cimetiere du Sud,20,0.0,0.0,100.0,100.0,4,0.0
149,lille,50.62,3.10,1,Hellu - Épine,14,0.0,0.0,100.0,100.0,4,0.0
208,lille,50.67,3.17,1,Gustave Delory,18,0.0,0.0,100.0,100.0,4,0.0
220,lille,50.69,3.16,1,Centre Nautique,10,0.0,0.0,100.0,100.0,4,0.0
239,lyon,45.70,4.80,1,SAINT GENIS - HOPITAUX SUD,13,0.0,0.0,100.0,100.0,3,0.0
245,lyon,45.71,4.81,1,OULLINS - MAIRIE,16,0.0,0.0,100.0,100.0,3,0.0
297,lyon,45.76,4.80,4,5055 - BUYER / APOLLINAIRE,66,0.0,0.0,100.0,100.0,12,0.0
298,lyon,45.76,4.81,2,5030 - CHARCOT / RÉSIDENCE UNIVERSITAIRE,46,0.0,0.0,100.0,100.0,6,0.0
325,lyon,45.78,4.78,1,ECULLY - MAIRIE,9,0.0,0.0,100.0,100.0,3,0.0
326,lyon,45.78,4.79,1,9050 - ALLEE DE L'EUROPE,10,0.0,0.0,100.0,100.0,3,0.0


## Question 3 : Quels sont les pics d’utilisation journaliers du réseau (et évolution mensuelle) ?

In [16]:
if USE_CLICKHOUSE:
    peaks = wh.query_df('''
        SELECT
            ville_id,
            heure_locale,
            if(type_jour = 'weekend', 'Week-end', 'Semaine') AS type_jour,
            round(taux_utilisation_moyen * 100, 1) AS taux_utilisation_pct,
            round(pct_stations_vides * 100, 1) AS pct_stations_vides,
            round(pct_stations_pleines * 100, 1) AS pct_stations_pleines,
            nb_releves
        FROM urbanhub.v_velo_pics_horaires
        ORDER BY ville_id, type_jour, heure_locale
    ''')
    monthly = wh.query_df('''
        SELECT
            ville_id,
            formatDateTime(mois_utc, '%Y-%m') AS mois,
            round(taux_utilisation_moyen * 100, 1) AS taux_utilisation_pct,
            round(pct_stations_critiques * 100, 1) AS pct_critiques,
            nb_releves
        FROM urbanhub.v_velo_tendance_mensuelle
        ORDER BY mois, ville_id
    ''')
else:
    peaks = daily_usage_peaks(df_local, tz='Europe/Paris')
    monthly = pd.DataFrame()

fig3 = px.line(
    peaks,
    x='heure_locale',
    y='taux_utilisation_pct',
    color='ville_id',
    line_dash='type_jour',
    markers=True,
    title="Pics journaliers d'utilisation (heure locale Europe/Paris)",
    labels={'heure_locale': 'Heure (0-23h)', 'taux_utilisation_pct': "Taux d'utilisation (%)"},
)
fig3.update_xaxes(dtick=1, range=[0, 23])
fig3.show()

if not monthly.empty and monthly['mois'].nunique() > 1:
    fig3b = px.line(
        monthly,
        x='mois',
        y='taux_utilisation_pct',
        color='ville_id',
        markers=True,
        title="Évolution mensuelle du taux d'utilisation (historique ClickHouse)",
    )
    fig3b.show()

peaks.sort_values('taux_utilisation_pct', ascending=False).head(15)

,ville_id,type_jour,heure_locale,taux_utilisation_moyen,taux_occupation_moyen,velos_dispo_moyens,pct_stations_vides,pct_stations_pleines,nb_releves,taux_utilisation_pct,taux_occupation_pct
4,marseille,Semaine,16,1.000000,0.000000,0.000000,100.0,0.0,492,100.0,0.0
5,montpellier,Semaine,16,0.871936,0.128064,1.890625,27.1,0.0,192,87.2,12.8
3,lyon,Semaine,16,0.744785,0.255215,5.055556,23.0,2.0,1368,74.5,25.5
1,clermont-ferrand,Semaine,16,0.718332,0.281668,5.838415,7.3,0.0,328,71.8,28.2
2,lille,Semaine,16,0.673801,0.326199,6.417925,8.5,0.4,1060,67.4,32.6
8,rennes,Semaine,16,0.659964,0.340036,8.100877,3.5,0.0,228,66.0,34.0
0,bordeaux,Semaine,16,0.655892,0.344108,6.796804,6.7,0.6,876,65.6,34.4
11,toulouse,Semaine,16,0.649634,0.350366,5.894273,9.9,2.3,1816,65.0,35.0
7,paris,Semaine,16,0.606554,0.393446,12.522582,7.1,3.2,4539,60.7,39.3
10,strasbourg,Semaine,16,0.603635,0.396365,5.075000,10.0,0.0,160,60.4,39.6


## Question 4 : Existe-t-il des déséquilibres géographiques dans la disponibilité des vélos ?

In [17]:
if USE_CLICKHOUSE:
    desequilibre = wh.query_df('''
        SELECT
            ville_id,
            secteur_geo,
            nb_stations,
            round(taux_occupation_moyen * 100, 1) AS taux_occupation_pct,
            round(pct_temps_vide * 100, 1) AS pct_temps_vide,
            round(pct_temps_pleine * 100, 1) AS pct_temps_pleine,
            distance_centre_km
        FROM urbanhub.v_velo_desequilibre_geo
    ''')
    city_avg = desequilibre.groupby('ville_id')['taux_occupation_pct'].transform('mean')
    desequilibre['ecart_moyenne_ville_pts'] = (desequilibre['taux_occupation_pct'] - city_avg).round(1)
    desequilibre = desequilibre.sort_values(['ville_id', 'ecart_moyenne_ville_pts'])
else:
    desequilibre = geographic_imbalance(df_local, city_centers=city_centers, center_radius_km=1.5)

fig4 = px.bar(
    desequilibre,
    x='secteur_geo',
    y='ecart_moyenne_ville_pts',
    color='ville_id',
    barmode='group',
    title='Écart de disponibilité par secteur géographique (Centre vs quadrants périphériques)',
    labels={'secteur_geo': 'Secteur', 'ecart_moyenne_ville_pts': 'Écart vs moyenne ville (pts %)'},
)
fig4.add_hline(y=0, line_dash='dash', line_color='gray')
fig4.show()
desequilibre

,ville_id,secteur_geo,nb_stations,distance_moyenne_km,velos_dispo_moyens,taux_occupation_moyen,taux_utilisation_moyen,pct_temps_vide,pct_temps_pleine,ecart_type_occupation,taux_occupation_pct,taux_utilisation_pct,ecart_moyenne_ville_pts
0,bordeaux,Centre (<1.5 km),57,0.88,5.1,0.267925,0.732075,7.9,0.0,0.188,26.8,73.2,-7.8
4,bordeaux,Sud-Ouest,34,4.10,6.2,0.308944,0.691056,9.6,2.2,0.263,30.9,69.1,-3.7
2,bordeaux,Nord-Ouest,37,3.68,6.1,0.336715,0.663285,0.0,0.0,0.185,33.7,66.3,-0.9
1,bordeaux,Nord-Est,55,3.22,8.6,0.407270,0.592730,7.3,0.9,0.234,40.7,59.3,6.1
3,bordeaux,Sud-Est,36,2.95,7.9,0.409042,0.590958,8.3,0.0,0.234,40.9,59.1,6.3
8,clermont-ferrand,Sud-Ouest,2,2.15,3.5,0.225950,0.774050,0.0,0.0,0.159,22.6,77.4,-3.2
6,clermont-ferrand,Nord-Est,14,2.89,5.0,0.231120,0.768880,14.3,0.0,0.145,23.1,76.9,-2.8
5,clermont-ferrand,Centre (<1.5 km),30,0.94,5.2,0.232457,0.767543,10.0,0.0,0.180,23.2,76.8,-2.7
7,clermont-ferrand,Sud-Est,36,5.35,6.9,0.345431,0.654569,2.8,0.0,0.159,34.5,65.5,8.6
10,lille,Nord-Est,119,7.63,5.1,0.298843,0.701157,8.8,0.8,0.221,29.9,70.1,-5.2


## Question 5 : Détection des stations critiques nécessitant un rééquilibrage

In [18]:
if USE_CLICKHOUSE:
    critiques = wh.query_df('''
        SELECT
            ville_id,
            station_id,
            station_name,
            latitude,
            longitude,
            dernier_timestamp AS timestamp,
            bikes_available,
            free_slots,
            capacity,
            round(dernier_taux_occupation * 100, 1) AS taux_occupation_pct,
            action_reequilibrage,
            delta_velos_equilibre,
            round(pct_temps_critique * 100, 1) AS pct_temps_critique,
            round(abs(dernier_taux_occupation - 0.5) * 100 + pct_temps_critique * 35 + least(capacity, 50) * 0.3, 1) AS score_priorite
        FROM urbanhub.v_velo_stations_critiques
        WHERE action_reequilibrage != 'AUCUNE'
        ORDER BY score_priorite DESC
    ''')
else:
    critiques = detect_critical_stations(df_local, low_threshold=0.10, high_threshold=0.90)

print('Répartition des actions recommandées :')
print(critiques['action_reequilibrage'].value_counts())

fig5 = px.scatter_map(
    critiques,
    lat='latitude',
    lon='longitude',
    color='action_reequilibrage',
    size='capacity',
    color_discrete_map={'REAPPROVISIONNER': '#d62728', 'DESENGORGER': '#1f77b4'},
    hover_name='station_name',
    hover_data=['ville_id', 'bikes_available', 'free_slots', 'delta_velos_equilibre', 'score_priorite'],
    map_style='carto-positron',
    zoom=5,
    height=520,
    title='Stations critiques nécessitant une intervention de rééquilibrage',
)
fig5.show()
critiques.head(20)

Répartition des actions recommandées :
action_reequilibrage
REAPPROVISIONNER    927
DESENGORGER         209
Name: count, dtype: int64


,ville_id,station_id,station_name,latitude,longitude,timestamp,bikes_available,free_slots,capacity,taux_occupation_pct,statut_criticite,action_reequilibrage,delta_velos_equilibre,pct_temps_critique,score_priorite
2674,paris,f72384ff88bee74a7fb3da49e1367f16,Concorde - Place de la Madeleine,48.870621,2.324220,2026-11-20 15:11:24.471945+00:00,0,400,400,0.0,CRITIQUE_VIDE,REAPPROVISIONNER,200,100.0,100.0
1279,paris,08cd405dc9467f0d86e2b6717e1382d3,Raspail - Bac,48.855629,2.325660,2026-11-20 15:11:24.324414+00:00,0,52,52,0.0,CRITIQUE_VIDE,REAPPROVISIONNER,26,100.0,100.0
2486,paris,d5f5b73d5f0c3069a29370f1b343f0ac,Herschel - Observatoire,48.843424,2.337665,2026-11-20 15:11:24.325109+00:00,0,56,56,0.0,CRITIQUE_VIDE,REAPPROVISIONNER,28,100.0,100.0
2584,paris,e7014080091445e46358f6b8c994fabd,Hector Malot - Daumesnil,48.846936,2.377088,2026-11-20 15:11:24.453044+00:00,54,0,54,100.0,CRITIQUE_PLEINE,DESENGORGER,-27,100.0,100.0
1551,paris,37a2ba913c8152ebeddb3228fb210d15,Filles Saint-Thomas - Place de la Bourse,48.868900,2.339754,2026-11-20 15:11:24.473342+00:00,0,54,54,0.0,CRITIQUE_VIDE,REAPPROVISIONNER,27,100.0,100.0
1728,paris,56c2da4066fd35120072927ac72d88b2,Place Valhubert,48.844183,2.364037,2026-11-20 15:11:24.530437+00:00,51,0,51,100.0,CRITIQUE_PLEINE,DESENGORGER,-25,100.0,100.0
1645,paris,4837dabfb36f1567edcef37a72bf4a22,Station Tour de France,48.883291,2.334151,2026-11-20 15:11:24.601977+00:00,0,200,200,0.0,CRITIQUE_VIDE,REAPPROVISIONNER,100,100.0,100.0
2418,paris,cd8520e3523db2cabae509e756e9c13c,Mézières - Rennes,48.850235,2.330382,2026-11-20 15:11:24.451407+00:00,55,0,55,100.0,CRITIQUE_PLEINE,DESENGORGER,-27,100.0,100.0
1119,marseille,d74960ffcea6c80ec1ed6d681c9d259c,Escale Borély,43.254948,5.376452,2026-11-20 15:14:28.575564+00:00,0,60,60,0.0,CRITIQUE_VIDE,REAPPROVISIONNER,30,100.0,100.0
2472,paris,d3bd04f60bec2a43506568b72170354f,Invalides - Quai d'Orsay,48.862523,2.310176,2026-11-20 15:11:24.604418+00:00,0,400,400,0.0,CRITIQUE_VIDE,REAPPROVISIONNER,200,100.0,100.0


## Synthèse & recommandations opérationnelles

- **Planification du rééquilibrage** : prioriser les stations à fort `score_priorite` en déplaçant les vélos des stations à `DESENGORGER` (delta négatif) vers les stations proches à `REAPPROVISIONNER` (delta positif).
- **Dimensionnement de l'offre** : les zones identifiées en Question 2 présentant un taux de sous-offre persistant (> 50 % du temps sous 15 % de vélos) sont candidates à une extension de capacité ou à l'ajout de stations.
- **Anticipation des pics** : programmer les tournées de régulation 30 à 45 minutes avant les pics pendulaires détectés en Question 3 (généralement 8h-9h et 17h30-19h en semaine).
- **Couverture historique** : l'archive officielle CityBikes (`data.citybik.es`, ODbL v1.0) débute en novembre 2024 et est complétée en continu par le flux streaming toutes les 2-3 minutes (`api.citybik.es/v2`).